# Import necessary libraries

In [19]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from joblib import dump, load
from tabulate import tabulate

# Load dataset

In [20]:
df = pd.read_csv('../data/training_data.csv')
df.head()

,Diabetes,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,0.0,1.0,1.0,15.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,5.0,10.0,20.0,0.0,0.0,11.0,4.0,5.0
1,1.0,1.0,0.0,1.0,28.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,4.0,3.0
2,1.0,1.0,1.0,1.0,33.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,2.0,10.0,0.0,0.0,0.0,9.0,4.0,7.0
3,1.0,0.0,1.0,1.0,29.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,5.0,0.0,30.0,1.0,1.0,12.0,3.0,4.0
4,0.0,0.0,0.0,1.0,24.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,3.0,0.0,0.0,1.0,1.0,13.0,5.0,6.0


# Define training-related stuffs

In [21]:
LABEL_COL = 'Diabetes'
CATEGORICAL_COLS = ['GenHlth', 'Age', 'Education', 'Income']
NUMERICAL_COLS = ['BMI', 'MentHlth', 'PhysHlth']

NUM_FOLDS = 5
LR = 1e-2
L2_REG = 1e-4
MAX_DEPTH = 20
NUM_EPOCHS = 1000
PATIENCE = 10
MIN_CHILD_WEIGHT = 8
COSAMPLE_RATE = 0.8
DATA_RATE = 0.8

In [22]:
# Best so far

# NUM_FOLDS = 5
# LR = 1e-3
# L2_REG = 1e-4
# MAX_DEPTH = 20
# NUM_EPOCHS = 1000
# PATIENCE = 20
# MIN_CHILD_WEIGHT = 5
# COSAMPLE_RATE = 0.8
# DATA_RATE = 0.8

# Data preprocessing

In [23]:
# Separate features and labels
X = df.drop(columns=[LABEL_COL])
y = df[LABEL_COL]

# Store one encoder per column
encoders = {}
for col in CATEGORICAL_COLS:
    encoder = LabelEncoder()
    X[col] = encoder.fit_transform(X[col])
    encoders[col] = encoder  # Save each encoder with its column name
# Save all encoders as a dictionary
dump(encoders, '../models/label_encoders.bin')

# How to load encoder
# # Load all encoders
# encoders = load('../models/label_encoders.bin')
# # Transform new data
# for col in CATEGORICAL_COLS:
#     X_new[col] = encoders[col].transform(X_new[col])
    
# Normalize numerical features
scaler = StandardScaler()
X[NUMERICAL_COLS] = scaler.fit_transform(X[NUMERICAL_COLS])
# Save the scaler for future use
dump(scaler, '../models/scaler.bin')

# How to load scaler
# # Load scaler
# scaler = load('../models/scaler.bin')
# # Transform new data
# X_new[NUMERICAL_COLS] = scaler.transform(X_new[NUMERICAL_COLS])

# Calculate scale_pos_weight for class imbalance
scale_pos_weight = (y == 0).sum() / (y == 1).sum()
print(f"\nScale pos weight: {scale_pos_weight:.4f}")


Scale pos weight: 4.2981


# K-Fold Cross Validation

In [24]:
# K-fold Cross Validation Training
kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)
fold = 1 # Fold counter
classification_reports = [] # To store classification reports for each fold
best_iterations = [] # To store best iterations for each fold

for train_index, val_index in kf.split(X):
    print(f"Training fold {fold}...")
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    
    # Base model
    model = xgb.XGBClassifier(
        learning_rate=LR,
        max_depth=MAX_DEPTH,
        n_estimators=NUM_EPOCHS,
        reg_lambda=L2_REG,  # L2 regularization
        eval_metric='logloss',
        early_stopping_rounds=PATIENCE,
        enable_categorical=False,  # Categorical features are already encoded
        scale_pos_weight=scale_pos_weight,  # Handle class imbalance
        random_state=42,
        min_child_weight = MIN_CHILD_WEIGHT, # Minimum sum of instance weight needed in a child
        colsample_bytree = COSAMPLE_RATE, # Subsample ratio of columns when constructing each tree
        subsample = DATA_RATE # Subsample ratio of the training instances
    )
    
    # Train the model
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              verbose=False)
    
    # Store best iteration
    best_iterations.append(model.get_booster().best_iteration)
    print(f"Best iteration for fold {fold}: {model.get_booster().best_iteration}")
    
    # Evaluate the model
    y_pred = model.predict(X_val)
    cr = classification_report(y_val, y_pred, output_dict=True)
    cm = confusion_matrix(y_val, y_pred)
    print(f"Fold {fold} results:")
    print(cr)
    print("Confusion Matrix:")
    print(cm)
    
    # Store report
    classification_reports.append(cr)
    
    print("\n")
    fold += 1

Training fold 1...
Best iteration for fold 1: 999
Fold 1 results:
{'0.0': {'precision': 0.8915081703723546, 'recall': 0.7765455144394422, 'f1-score': 0.8300652156370812, 'support': 72856.0}, '1.0': {'precision': 0.3864706990766912, 'recall': 0.5983080513418904, 'f1-score': 0.46960503720663993, 'support': 17140.0}, 'accuracy': 0.7425996710964932, 'macro avg': {'precision': 0.6389894347245229, 'recall': 0.6874267828906663, 'f1-score': 0.6498351264218606, 'support': 89996.0}, 'weighted avg': {'precision': 0.7953223148009106, 'recall': 0.7425996710964932, 'f1-score': 0.7614145260697921, 'support': 89996.0}}
Confusion Matrix:
[[56576 16280]
 [ 6885 10255]]


Training fold 2...
Best iteration for fold 2: 999
Fold 2 results:
{'0.0': {'precision': 0.894617599395837, 'recall': 0.7771505890714266, 'f1-score': 0.8317571768147742, 'support': 73166.0}, '1.0': {'precision': 0.38325074705904605, 'recall': 0.602020202020202, 'f1-score': 0.4683477014814986, 'support': 16830.0}, 'accuracy': 0.7443997511

# Cross-validation results

In [25]:
avg_report = {}

# Extract average metrics across folds
for key in classification_reports[0].keys():
    if key in ['0.0', '1.0', 'macro avg', 'weighted avg']: # Average the metric sections
        avg_report[key] = {}
        for metric in classification_reports[0][key].keys(): # Iterate through metrics ('precision', 'recall', etc.)
            avg_report[key][metric] = np.mean([report[key][metric] for report in classification_reports])
    elif key == 'accuracy':
        avg_report[key] = np.mean([report[key] for report in classification_reports])

# Format and print the average classification report
headers = ["precision", "recall", "f1-score", "support"]
table = []
for label in ['0.0', '1.0', 'macro avg', 'weighted avg']:
    if label in avg_report:
        row = [
            label,
            f"{avg_report[label]['precision']:.4f}",
            f"{avg_report[label]['recall']:.4f}",
            f"{avg_report[label]['f1-score']:.4f}",
            f"{int(avg_report[label]['support']):d}"
        ]
        table.append(row)

print(tabulate(table, headers=headers, floatfmt=".4f", numalign="right"))
print(f"\nAverage Accuracy: {avg_report['accuracy']:.4f}")

                precision    recall    f1-score    support
------------  -----------  --------  ----------  ---------
0.0                0.8925    0.7770      0.8307      73009
1.0                0.3841    0.5978      0.4677      16986
macro avg          0.6383    0.6874      0.6492      89995
weighted avg       0.7965    0.7432      0.7622      89995

Average Accuracy: 0.7432


# Train final model on entire dataset

In [26]:
# Train final model on entire dataset
final_model = xgb.XGBClassifier(
    learning_rate=LR,
    max_depth=MAX_DEPTH,
    n_estimators=int(np.mean(best_iterations)),
    reg_lambda=L2_REG,  # L2 regularization
    eval_metric='error',  # Classification error rate (1 - accuracy)
    scale_pos_weight=scale_pos_weight,  # Handle class imbalance
    random_state=42,
    min_child_weight = MIN_CHILD_WEIGHT, # Minimum sum of instance weight needed in a child
    colsample_bytree = COSAMPLE_RATE, # Subsample ratio of columns when constructing
    subsample = DATA_RATE # Subsample ratio of the training instances
)

final_model.fit(X, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='error', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.01, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=20,
              max_leaves=None, min_child_weight=8, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=998,
              n_jobs=None, num_parallel_tree=None, ...)

# Save the model

In [27]:
# Save the trained model
dump(final_model, '../models/xgboost_model_v2.bin')
print("\nModel saved to '../models/xgboost_model_v2.bin'")


Model saved to '../models/xgboost_model_v2.bin'
